# Modelado predictivo de segmentos de precio de vino

**Master:** Big Data & Business Intelligence 2025/2026  
**Asignatura:** Modelado Predictivo con Machine Learning  
**Integrantes:** [Anadir nombres del grupo]  
**Vinculacion con el TFM:** Extension exploratoria de Machine Learning del TFM sobre analisis de la oferta y comportamiento de precios del vino en los catalogos digitales de Walmart Costa Rica, Masxmenos Costa Rica y PriceSmart Costa Rica.

El TFM principal mantiene un enfoque descriptivo y de Business Intelligence, de acuerdo con el alcance definido con el tutor. Para mantener coherencia con ese enfoque, este notebook desarrolla un **modelo supervisado de clasificacion** que predice el `segmento_precio` de cada producto a partir de atributos del catalogo, sin plantear una regresion de precio como modelo principal.


## 1. Datos y justificacion

El conjunto es propio y procede del scraping diario de precios de vinos en supermercados costarricenses con presencia digital: Walmart Costa Rica, Masxmenos Costa Rica y PriceSmart Costa Rica. Esta es la misma fuente usada por el TFM para analizar oferta, comparacion de precios, evolucion de corto plazo y posicionamiento comercial.

La unidad de analisis es un registro de producto observado en una fecha y comercio. La variable objetivo del modelo supervisado es `segmento_precio`, una etiqueta categorica alineada con el TFM porque permite clasificar la oferta en niveles comerciales comparables.

El `precio_equivalente_750ml_crc` se conserva para analisis exploratorio, validacion de coherencia de segmentos y clustering, pero no se usa como variable predictora del modelo de clasificacion para evitar fuga de informacion.


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score, silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
PRICE_TARGET = 'precio_equivalente_750ml_crc'
CLASS_TARGET = 'segmento_precio'
MIN_PRICE_CRC = 500
OUTPUT_DIR = Path('resultados')
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
csv_path = Path('webscraping_precios_vino_clean (7).csv')
if not csv_path.exists():
    raise FileNotFoundError(f'No se encontro el CSV: {csv_path.resolve()}')

df_raw = pd.read_csv(csv_path, encoding='utf-8-sig')
df_raw['fecha_extraccion'] = pd.to_datetime(df_raw['fecha_extraccion'], errors='coerce')
for column in ['presentacion_ml', PRICE_TARGET, 'precio_final_crc', 'descuento_pct']:
    if column in df_raw.columns:
        df_raw[column] = pd.to_numeric(df_raw[column], errors='coerce')

print(f'Filas: {len(df_raw):,}; columnas: {df_raw.shape[1]}')
print(f'Periodo: {df_raw.fecha_extraccion.min().date()} a {df_raw.fecha_extraccion.max().date()}')
print(f'Comercios: {df_raw.retailer.nunique()}')
print(f'Segmentos etiquetados: {df_raw[CLASS_TARGET].notna().sum():,}')
display(df_raw.head())


## 2. Analisis exploratorio de datos

Se inspeccionan estructura, faltantes y cobertura por categoria/comercio/segmento. Las distribuciones de precio se muestran en CRC por 750 ml para comprender la coherencia de las etiquetas, pero el modelo supervisado elegido es de clasificacion de `segmento_precio`.


In [ ]:
display(df_raw.dtypes.rename('tipo').to_frame())
missing = df_raw.isna().sum().sort_values(ascending=False).rename('faltantes').to_frame()
display(missing)

price_valid = df_raw[PRICE_TARGET].notna() & (df_raw[PRICE_TARGET] >= MIN_PRICE_CRC)
eda_valid = df_raw.loc[price_valid].copy()
display(eda_valid.groupby('categoria', dropna=False)[PRICE_TARGET].agg(['count', 'median', 'mean', 'min', 'max']).round(2))
display(eda_valid.groupby('retailer', dropna=False)[PRICE_TARGET].agg(['count', 'median', 'mean']).round(2))
display(df_raw[CLASS_TARGET].value_counts(dropna=False).rename('registros').to_frame())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
eda_valid[PRICE_TARGET].plot(kind='hist', bins=35, ax=axes[0], color='#167d6a', edgecolor='white')
axes[0].set(title='Distribucion del precio equivalente', xlabel='CRC por 750 ml')
eda_valid.boxplot(column=PRICE_TARGET, by=CLASS_TARGET, ax=axes[1], rot=30, grid=False)
axes[1].set(title='Precio por segmento', xlabel='', ylabel='CRC por 750 ml')
fig.suptitle('')
plt.tight_layout()
plt.show()

daily_median = eda_valid.groupby(eda_valid['fecha_extraccion'].dt.date)[PRICE_TARGET].median()
ax = daily_median.plot(figsize=(10, 4), marker='o', color='#167d6a')
ax.set(title='Mediana diaria del precio equivalente', xlabel='Fecha', ylabel='CRC por 750 ml')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()


## 3. Preparacion y transformacion de variables

Se eliminan unicamente duplicados exactos. Los precios equivalentes menores a 500 CRC se apartan para revision manual, porque pueden afectar la interpretacion de segmentos y el analisis no supervisado. Para el modelo de clasificacion se usan solo filas con `segmento_precio` disponible.

TF-IDF transforma la descripcion del producto; OneHotEncoder codifica categoria y comercio; StandardScaler escala volumen, descuento y tiempo. El preprocesamiento se ajusta dentro del pipeline para evitar ajustar transformadores usando el conjunto de prueba.


In [ ]:
df = df_raw.drop_duplicates().copy()
suspicious = df.loc[df[PRICE_TARGET].isna() | (df[PRICE_TARGET] < MIN_PRICE_CRC)].copy()
clean = df.loc[df[PRICE_TARGET].notna() & (df[PRICE_TARGET] >= MIN_PRICE_CRC)].copy()
clean = clean.dropna(subset=['fecha_extraccion', 'presentacion_ml', 'producto', 'retailer', CLASS_TARGET])
clean[CLASS_TARGET] = clean[CLASS_TARGET].astype(str)
suspicious.to_csv(OUTPUT_DIR / 'precios_sospechosos_notebook.csv', index=False, encoding='utf-8-sig')
clean.to_csv(OUTPUT_DIR / 'datos_limpios_notebook.csv', index=False, encoding='utf-8-sig')
print(f'Duplicados exactos: {len(df_raw) - len(df)}')
print(f'Filas sospechosas apartadas: {len(suspicious)}')
print(f'Filas etiquetadas validas para clasificacion: {len(clean)}')
print(f'Segmentos: {sorted(clean[CLASS_TARGET].unique())}')
display(suspicious[['fecha_extraccion', 'producto', PRICE_TARGET]].head(10))


## 4. Modelo supervisado principal: clasificacion de segmento

Se eligio un modelo de clasificacion porque es la opcion mas coherente con el TFM: permite asignar cada producto a un segmento comercial (`economico`, `medio`, `premium` o `super premium`) y facilita el analisis de posicionamiento de la oferta.

El algoritmo principal es clasificador logistico multiclase. Es adecuado como baseline porque combina interpretabilidad, regularizacion y buen comportamiento con variables dispersas de texto. La regularizacion se ajusta mediante validacion temporal por bloques usando F1 macro, y la evaluacion final reserva las fechas mas recientes.


In [ ]:
def add_model_features(frame, start_date):
    result = frame.copy()
    result['dias_desde_inicio'] = (result['fecha_extraccion'] - start_date).dt.days
    result['unidades_750ml'] = result['presentacion_ml'] / 750
    result['producto'] = result['producto'].fillna('').astype(str)
    result['descuento_pct'] = result['descuento_pct'].fillna(0)
    for column in ['categoria', 'retailer']:
        result[column] = result[column].fillna('desconocido').astype(str)
    return result

def make_classification_pipeline(c_value):
    preprocessing = ColumnTransformer([
        ('texto', TfidfVectorizer(max_features=2500, ngram_range=(1, 2), min_df=2), 'producto'),
        ('categorias', OneHotEncoder(handle_unknown='ignore'), ['categoria', 'retailer']),
        ('numericas', StandardScaler(), ['unidades_750ml', 'dias_desde_inicio', 'descuento_pct']),
    ])
    return Pipeline([
        ('preparacion', preprocessing),
        ('modelo', LogisticRegression(C=c_value, max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)),
    ])

unique_dates = np.sort(clean['fecha_extraccion'].unique())
test_start = unique_dates[max(1, int(len(unique_dates) * 0.8))]
train_raw = clean.loc[clean['fecha_extraccion'] < test_start].copy()
test_raw = clean.loc[clean['fecha_extraccion'] >= test_start].copy()
anchor_date = clean['fecha_extraccion'].min()
train = add_model_features(train_raw, anchor_date)
test = add_model_features(test_raw, anchor_date)
class_names = sorted(train[CLASS_TARGET].unique())
print(f'Entrenamiento: hasta {train_raw.fecha_extraccion.max().date()} ({len(train):,} filas)')
print(f'Prueba: {test_raw.fecha_extraccion.min().date()} a {test_raw.fecha_extraccion.max().date()} ({len(test):,} filas)')
print(f'Clases en entrenamiento: {class_names}')


In [ ]:
# Validacion de origen movil con bloques de fechas completos.
train_dates = np.sort(train['fecha_extraccion'].unique())
folds = []
first_validation = max(3, int(len(train_dates) * 0.55))
for start in range(first_validation, len(train_dates) - 1, 2):
    validation_dates = train_dates[start:start + 2]
    fold_train = np.flatnonzero(train['fecha_extraccion'].isin(train_dates[:start]))
    fold_valid = np.flatnonzero(train['fecha_extraccion'].isin(validation_dates))
    if len(fold_train) and len(fold_valid):
        folds.append((fold_train, fold_valid))
if not folds:
    raise ValueError('No hay suficientes fechas para realizar la validacion temporal.')

c_results = []
for c_value in [0.01, 0.1, 1.0, 3.0, 10.0]:
    fold_f1 = []
    for fit_indices, valid_indices in folds:
        fold_model = make_classification_pipeline(c_value)
        fold_model.fit(train.iloc[fit_indices], train.iloc[fit_indices][CLASS_TARGET])
        fold_prediction = fold_model.predict(train.iloc[valid_indices])
        fold_f1.append(f1_score(train.iloc[valid_indices][CLASS_TARGET], fold_prediction, average='macro', zero_division=0))
    c_results.append({'C': c_value, 'F1_macro_CV': np.mean(fold_f1)})

tuning_results = pd.DataFrame(c_results).sort_values('F1_macro_CV', ascending=False)
display(tuning_results)
best_c = float(tuning_results.iloc[0]['C'])
final_classifier = make_classification_pipeline(best_c)
final_classifier.fit(train, train[CLASS_TARGET])
class_prediction = final_classifier.predict(test)
majority_class = train[CLASS_TARGET].mode().iloc[0]
baseline_prediction = np.full(len(test), majority_class)

classification_results = pd.DataFrame([
    {
        'modelo': f'Base: clase mayoritaria ({majority_class})',
        'accuracy': accuracy_score(test[CLASS_TARGET], baseline_prediction),
        'F1_macro': f1_score(test[CLASS_TARGET], baseline_prediction, average='macro', zero_division=0),
        'filas_prueba': len(test),
    },
    {
        'modelo': f'Clasificador logistico multiclase C={best_c}',
        'accuracy': accuracy_score(test[CLASS_TARGET], class_prediction),
        'F1_macro': f1_score(test[CLASS_TARGET], class_prediction, average='macro', zero_division=0),
        'filas_prueba': len(test),
    },
])
display(classification_results.round(3))
classification_results.to_csv(OUTPUT_DIR / 'metricas_clasificacion_notebook.csv', index=False, encoding='utf-8-sig')

class_report = classification_report(test[CLASS_TARGET], class_prediction, labels=class_names, zero_division=0, output_dict=True)
class_report_df = pd.DataFrame(class_report).T
display(class_report_df.round(3))
class_report_df.to_csv(OUTPUT_DIR / 'reporte_clasificacion_notebook.csv', encoding='utf-8-sig')

predictions = test[['fecha_extraccion', 'categoria', 'producto', 'retailer', PRICE_TARGET, CLASS_TARGET]].copy()
predictions['prediccion_segmento'] = class_prediction
predictions['acierto'] = predictions[CLASS_TARGET] == predictions['prediccion_segmento']
predictions.to_csv(OUTPUT_DIR / 'predicciones_clasificacion_notebook.csv', index=False, encoding='utf-8-sig')
display(predictions.head(10))


## 5. Modelo avanzado o extension: clasificacion multiclase

La extension avanzada elegida es el enfoque **multiclase**: el modelo no separa solo dos grupos, sino cuatro segmentos comerciales. Se reporta F1 macro para que todas las clases tengan peso similar en la evaluacion, incluso si el dataset esta desbalanceado.

Esta extension aporta valor al TFM porque permite traducir atributos del catalogo en una categoria de posicionamiento comercial que puede compararse por comercio, categoria y fecha.


In [ ]:
best_model_row = classification_results.loc[classification_results['modelo'].str.startswith('Clasificador')].iloc[0]
print(f"Modelo elegido: {best_model_row['modelo']}")
print(f"Accuracy de prueba: {best_model_row['accuracy']:.3f}")
print(f"F1 macro de prueba: {best_model_row['F1_macro']:.3f}")
print(f"Filas de prueba: {int(best_model_row['filas_prueba'])}")

segment_summary = predictions.groupby([CLASS_TARGET, 'prediccion_segmento']).size().rename('filas').reset_index()
display(segment_summary)


## 6. Analisis no supervisado: segmentos de productos

Se agregan las observaciones por nombre de producto para que las repeticiones diarias no dominen el clustering. TF-IDF representa el texto, TruncatedSVD reduce su dimension y StandardScaler normaliza precio, volumen y frecuencia de observacion. KMeans se compara con silhouette para k entre 2 y 6.

Este apartado es independiente del modelo supervisado de clasificacion y cumple la parte no supervisada de la rubrica.


In [ ]:
products = (
    clean.groupby('producto', dropna=False)
    .agg(
        mediana_precio=(PRICE_TARGET, 'median'),
        mediana_presentacion=('presentacion_ml', 'median'),
        observaciones=(PRICE_TARGET, 'size'),
        categoria=('categoria', lambda values: values.mode().iloc[0] if not values.mode().empty else 'desconocido'),
    )
    .reset_index()
)
text_matrix = TfidfVectorizer(max_features=1000, min_df=2, ngram_range=(1, 2)).fit_transform(products['producto'].fillna(''))
if min(text_matrix.shape) < 3:
    raise ValueError('No hay variedad textual suficiente para segmentar los productos.')
svd_components = min(20, text_matrix.shape[0] - 1, text_matrix.shape[1] - 1)
text_embedding = TruncatedSVD(n_components=svd_components, random_state=RANDOM_STATE).fit_transform(text_matrix)
numeric_cluster = products[['mediana_precio', 'mediana_presentacion', 'observaciones']].copy()
numeric_cluster['mediana_precio'] = np.log1p(numeric_cluster['mediana_precio'])
numeric_scaled = StandardScaler().fit_transform(numeric_cluster)
cluster_features = np.hstack([text_embedding, numeric_scaled])

cluster_scores = []
for k in range(2, min(7, len(products))):
    candidate = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    cluster_labels = candidate.fit_predict(cluster_features)
    cluster_scores.append({'k': k, 'silhouette': silhouette_score(cluster_features, cluster_labels)})
scores = pd.DataFrame(cluster_scores).sort_values('silhouette', ascending=False)
display(scores)
scores.to_csv(OUTPUT_DIR / 'silhouette_clusters_notebook.csv', index=False, encoding='utf-8-sig')
best_k = int(scores.iloc[0]['k'])
products['cluster'] = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10).fit_predict(cluster_features)
products.to_csv(OUTPUT_DIR / 'segmentacion_productos_notebook.csv', index=False, encoding='utf-8-sig')
cluster_profile = products.groupby('cluster').agg(
    productos=('producto', 'count'),
    precio_mediano=('mediana_precio', 'median'),
    volumen_mediano_ml=('mediana_presentacion', 'median'),
    observaciones_mediana=('observaciones', 'median'),
).round(2)
display(cluster_profile)
cluster_profile.to_csv(OUTPUT_DIR / 'perfil_clusters_notebook.csv', encoding='utf-8-sig')

fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(products['mediana_presentacion'], products['mediana_precio'], c=products['cluster'], cmap='viridis', alpha=0.75)
ax.set(title='Segmentos de productos por volumen y precio', xlabel='Volumen mediano (ml)', ylabel='Precio mediano equivalente (CRC)')
fig.colorbar(scatter, ax=ax, label='Cluster')
plt.tight_layout()
plt.show()

## 7. Conclusiones, limitaciones y trabajo futuro

Se eligio la opcion de **modelo de clasificacion**. El modelo principal predice `segmento_precio` mediante clasificador logistico multiclase, usando texto del producto, categoria, comercio, volumen, descuento y fecha. Esta decision es coherente con el TFM porque el segmento de precio forma parte del analisis de posicionamiento comercial de la oferta.

La lectura debe ser prudente: el primer avance del TFM delimita el proyecto principal como descriptivo y de Business Intelligence, porque la serie historica todavia es corta. Por ello, las metricas del clasificador no deben presentarse como evidencia definitiva de capacidad predictiva del mercado, sino como una exploracion inicial sobre la base de datos construida.

Limitaciones conocidas: periodo de observacion corto; errores y variantes en titulos/categorias; 46 registros iniciales con precios equivalentes menores a 500 CRC aislados por un umbral provisional; posible error de volumen/unidades en paquetes; etiquetas de segmento incompletas; y presencia de productos repetidos entre fechas. La prueba temporal no demuestra rendimiento sobre productos nuevos.

Trabajo futuro: completar los nombres del grupo, validar manualmente anomalias y equivalencias de volumen, normalizar catalogo, revisar los umbrales usados para `segmento_precio`, comparar clasificadores y robustecer la validacion temporal. Si el TFM acumula suficiente historia, el clasificador podria evolucionar hacia un componente estable de apoyo al dashboard.
